<a href="https://colab.research.google.com/github/Kongbeng-21/SuperAi/blob/main/word_segmentation_baseline_template.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


# Setup

In [ ]:
# โจทย์นี้ไม่ต้องใช้ API key
# และ baseline นี้ไม่ต้องติดตั้งแพ็กเกจเพิ่ม
print("✓ Setup เสร็จแล้ว")


# Import and Config

In [ ]:
from pathlib import Path
from collections import Counter
import math
import pandas as pd

INPUT_ROOT = Path('/kaggle/input')
OUTPUT_PATH = Path('/kaggle/working/submission.csv')

TEST_FILENAME = 'ws_test.txt'
SAMPLE_SUB_FILENAME = 'ws_sample_submission.csv'
TRAIN_DIRNAME = 'train'
EVAL_DIRNAME = 'eval'

print('✓ Import และ Config เสร็จแล้ว')


# Load Data

In [ ]:
def find_file(filename: str):
    for p in INPUT_ROOT.rglob(filename):
        return p
    return None

def find_dir(dirname: str):
    for p in INPUT_ROOT.rglob(dirname):
        if p.is_dir():
            return p
    return None

ws_test_path = find_file(TEST_FILENAME)
sample_sub_path = find_file(SAMPLE_SUB_FILENAME)
train_dir = find_dir(TRAIN_DIRNAME)
eval_dir = find_dir(EVAL_DIRNAME)

print('ws_test_path =', ws_test_path)
print('sample_sub_path =', sample_sub_path)
print('train_dir =', train_dir)
print('eval_dir =', eval_dir)

if ws_test_path is None or sample_sub_path is None or train_dir is None or eval_dir is None:
    raise FileNotFoundError('หา ws_test / ws_sample_submission / train / eval ไม่เจอใน /kaggle/input')

with open(ws_test_path, 'r', encoding='utf-8-sig') as f:
    ws_test_text = f.read()

sample_sub = pd.read_csv(sample_sub_path)

print('len(ws_test_text) =', len(ws_test_text))
print('sample submission rows =', len(sample_sub))
display(sample_sub.head(10))


# Build Lexicon

In [ ]:
# สร้างพจนานุกรมคำจาก LST20 train + eval
# แต่จะไม่เอา '_' เพราะตัวนี้คือช่องว่างใน corpus

def iter_tokens_from_lst20(file_path: Path):
    with open(file_path, 'r', encoding='utf-8-sig') as f:
        for line in f:
            line = line.rstrip('\n')
            if not line.strip():
                continue
            parts = line.split('\t')
            if not parts:
                continue
            token = parts[0]
            yield token

word_freq = Counter()
char_freq = Counter()

for corpus_dir in [train_dir, eval_dir]:
    for file_path in sorted(corpus_dir.glob('*.txt')):
        for token in iter_tokens_from_lst20(file_path):
            if token == '_':
                continue
            token = token.strip()
            if not token:
                continue
            word_freq[token] += 1
            for ch in token:
                char_freq[ch] += 1

max_word_len = max(len(w) for w in word_freq)
total_words = sum(word_freq.values())

print('vocab size =', len(word_freq))
print('total words =', total_words)
print('max word len =', max_word_len)
print('top words =', word_freq.most_common(20))


# Segmenter

In [ ]:
# ใช้ dynamic programming แบบให้คะแนนด้วยความถี่คำ
# เพื่อให้ได้ segmentation ที่นิ่งกว่าการตัดแบบ longest match ตรง ๆ

MIN_WORD_PROB = 1e-12

def word_score(word: str) -> float:
    freq = word_freq.get(word, 0)
    if freq > 0:
        return math.log(freq / total_words)

    # fallback สำหรับคำที่ไม่อยู่ใน vocab
    # ลงโทษคำยาวที่ไม่รู้จัก แต่ยังให้ตัดผ่านได้
    char_bonus = sum(math.log((char_freq.get(ch, 1)) / max(sum(char_freq.values()), 1)) for ch in word)
    return math.log(MIN_WORD_PROB) * max(len(word), 1) + 0.2 * char_bonus

def segment_chunk(text: str):
    n = len(text)
    if n == 0:
        return []

    dp = [-10**18] * (n + 1)
    back = [-1] * (n + 1)
    dp[0] = 0.0

    for i in range(n):
        if dp[i] <= -10**17:
            continue

        limit = min(n, i + max_word_len)
        matched = False

        for j in range(i + 1, limit + 1):
            word = text[i:j]
            if word in word_freq:
                matched = True
                score = dp[i] + word_score(word)
                if score > dp[j]:
                    dp[j] = score
                    back[j] = i

        if not matched:
            # ถ้าไม่มีคำในพจนานุกรม ให้ลองหลายความยาวสั้น ๆ เพื่อให้ระบบยังเดินต่อได้
            for j in range(i + 1, min(n, i + 6) + 1):
                word = text[i:j]
                score = dp[i] + word_score(word)
                if score > dp[j]:
                    dp[j] = score
                    back[j] = i

    words = []
    idx = n
    while idx > 0:
        prev = back[idx]
        if prev == -1:
            prev = idx - 1
        words.append(text[prev:idx])
        idx = prev
    words.reverse()
    return words

def segment_text_preserve_whitespace(text: str):
    pieces = []
    current = []

    def flush_current():
        nonlocal current
        if current:
            chunk = ''.join(current)
            pieces.extend(segment_chunk(chunk))
            current = []

    for ch in text:
        if ch.isspace():
            flush_current()
            pieces.append(ch)
        else:
            current.append(ch)

    flush_current()
    return pieces

def words_to_bie_labels(words_or_spaces):
    labels = []
    for item in words_or_spaces:
        if item.isspace():
            continue
        if len(item) == 1:
            labels.append('B_WORD')
        elif len(item) == 2:
            labels.extend(['B_WORD', 'E_WORD'])
        else:
            labels.append('B_WORD')
            labels.extend(['I_WORD'] * (len(item) - 2))
            labels.append('E_WORD')
    return labels

segmented_preview = segment_text_preserve_whitespace(ws_test_text[:300])
print(segmented_preview[:80])


# Run

In [ ]:
segmented = segment_text_preserve_whitespace(ws_test_text)
predicted_labels = words_to_bie_labels(segmented)

print('predicted label count =', len(predicted_labels))
print('expected row count =', len(sample_sub))

if len(predicted_labels) != len(sample_sub):
    raise ValueError(f'จำนวน label ไม่ตรงกับ sample submission: got {len(predicted_labels)} vs expected {len(sample_sub)}')

submission = sample_sub.copy()
submission['Predicted'] = predicted_labels

submission.to_csv(OUTPUT_PATH, index=False)

print('✓ บันทึก submission แล้วที่', OUTPUT_PATH)
print(submission['Predicted'].value_counts())
display(submission.head(15))
